In [2]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store - returns list of tuples (metadata, distance)
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            for rank, (metadata, distance) in enumerate(results, 1):
                similarity_score = distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': rank,
                        'content': metadata.get('content', ''),
                        'source': metadata.get('source', ''),
                        'similarity_score': float(similarity_score),
                        'metadata': metadata,
                        'rank': rank
                    })
            
            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print(f"No documents retrieved above the score threshold of {score_threshold}")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

/Users/merlindconstanzepohl/Desktop/RAG Implementierung/Code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 55882.13it/s]


Loaded embedding model: BAAI/bge-m3. Embedding dimension: 1024
EmbeddingManager initialized successfully
Loaded index with 251 chunks
Vector store loaded successfully with 251 chunks
Retrieving documents for query: 'What is the economic contribution of the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


Retrieved 5 documents above the score threshold of 0.0

Rank 1: 0.5855
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: 44 7. TOTAL ECONOMIC VALUE OF CATCHMENT AREA The total economic contributions of tourism, commercial fishing and cultural & recreational activity to the GBRCA are the sums of the direct contributions ...

Rank 2: 0.5803
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: They are also, at present at least, largely masked by broader economic drivers such as real income growth, the value of the Australian dollar, oil prices and associated effects on airfares, geopolitic...

Rank 3: 0.5795
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: Access Economics notes that estimates of the economic cost of damage to the have already been made. One of these puts th

In [3]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 73482.07it/s]


Loaded embedding model: BAAI/bge-m3. Embedding dimension: 1024
EmbeddingManager initialized successfully
Loaded index with 251 chunks
Vector store loaded successfully with 251 chunks


In [4]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


Retrieving documents for query: 'What is the economic contribution of the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.00it/s]

Retrieved 5 documents above the score threshold of 0.0

Rank 1: 0.5855
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: 44 7. TOTAL ECONOMIC VALUE OF CATCHMENT AREA The total economic contributions of tourism, commercial fishing and cultural & recreational activity to the GBRCA are the sums of the direct contributions ...

Rank 2: 0.5803
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: They are also, at present at least, largely masked by broader economic drivers such as real income growth, the value of the Australian dollar, oil prices and associated effects on airfares, geopolitic...

Rank 3: 0.5795
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: Access Economics notes that estimates of the economic cost of damage to the have already been made. One of these puts th

In [5]:
#4. LLM integration with Ollama

from langchain_ollama import OllamaLLM
from dotenv import load_dotenv
load_dotenv()

#temperature defines how "creative" the model's answers will be --> low value for solid and factual answers
#top_p defines the diversity of the output ("creativity") --> low value for more deterministic answers (better not 0.0 --> often ignored and set to default)
llm = OllamaLLM(model="llama3", temperature=0.1, top_p=0.1)


In [6]:
#5. RAG function for information retrieval with minimal instructions


def retrieval_query(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, score_threshold: float = 0.3, return_context=False):

    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)
    
    if not results:
        return "No relevant documents found to answer the question."

    confidence = max(doc['similarity_score'] for doc in results)

    sources = []
    for doc in results:
        clean_preview = doc['content'][:150].replace('\n', ' ').strip()
        sources.append({
            'source': doc['source'],
            'score': round(float(doc['similarity_score']), 3),
            'preview': f"{clean_preview}..."
        })
    
    
    # in case of low confidence,choose to return the sources with a note about low confidence instead of an answer to avoid errors
    if confidence < score_threshold:
        return {
            'answer': "The found documents are not sufficiently relevant. I refuse to answer to avoid errors.",
            'sources': sources,
            'confidence': round(float(confidence), 3)
        }
    
    context = "\n\n".join([doc['content'] for doc in results])
    
    # generate answer    
    prompt = f"""
    Use the following context to answer the question concisely and factually. 
    Do not say where the information comes from, just give the answer. 
    If the provided texts mention different numbers or information for the same topic, list them separately. 

        Context: {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt)

    output = {
        'response': response,
        'sources': sources,
        'confidence': round(float(confidence), 3)
    }
    
    if return_context:
        output['context'] = context
    
    return output
 

In [7]:
#6. Output query with sources and confidence score

result = retrieval_query("How much will the Earth still warm up?", rag_retriever, llm, return_context=True)

print("\n" + "_"*100)
print("")
print("Answer:")
print("")
print(result['response'])

print("\n" + "_"*100)
print("")
print("Sources:")
print("")
for i, source in enumerate(result['sources'], 1):
    print(f"\n[Source {i}]")
    print(f"File: {source['source']}")
    print(f"Score: {source['score']}")
    print(f"Preview:{source['preview']}")

print("\n" + "_"*100)
print(f"Confidence Score: {result['confidence']}")
print("_"*100)

Retrieving documents for query: 'How much will the Earth still warm up?' with top_k=5 and score_threshold=0.3


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.47it/s]

Retrieved 5 documents above the score threshold of 0.3



____________________________________________________________________________________________________

Answer:

According to the provided context:

* Since the start of the 20th century, the global average surface temperature has risen between 0.6 and 0.7 degrees Celsius.
* Since then, the global average surface temperature has increased by 0.18 degrees Celsius per decade.
* In the 1980s, the increase in average global surface temperatures was even faster.

As for future warming, there is no specific prediction mentioned in the context. However, it is noted that some projections suggest the Arctic Ocean could be ice-free or nearly so by the end of the 21st century.

____________________________________________________________________________________________________

Sources:


[Source 1]
File: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Score: 0.629
Preview:As part of this focus, one related question is: how well can the adapt to cl

In [14]:
# 7. evaluation function for test queries with unshortened chunks (evaluation of recall@k and precision@k)

def evaluate_test_queries(queries: List[str], retriever: RAGRetriever, llm: OllamaLLM, filename="data/evaluation/evaluation__all_chunks_output.txt"):
    print("________ printed chunks _________")
    output_text = "" 
    
    with open(filename, 'w', encoding='utf-8') as f:
        # retrieve chunks
        for i, query in enumerate(queries, 1): 
            print(f"Question {i}: {query}\n")
            
            result = retrieval_query(query, retriever, llm, top_k=5, score_threshold=0.0, return_context=True)
            
            print("Generated answer:")
            print("_" * 40)
            print(result['response'] if isinstance(result, dict) else result)
            print("_" * 40)
            
            # adds question
            output_text += f"\n{'='*80}\n"
            output_text += f"Question {i}:\n"
            output_text += f"{query}\n"
            output_text += f"{'='*80}\n\n"
            
            # adds answer
            output_text += "Answer:\n"
            output_text += "-" * 40 + "\n"
            output_text += (result['response'] if isinstance(result, dict) else str(result)) + "\n"
            output_text += "-" * 40 + "\n\n"
            
            # adds retrieved chunks
            output_text += "Retrieved chunks:\n"
            output_text += "=" * 40 + "\n"
            
            if 'context' in result:
                chunks = result['context'].split("\n\n")
                for j, chunk in enumerate(chunks, 1):
                    output_text += f"\n[Chunk {j}]\n"
                    if j-1 < len(result['sources']):
                        src = result['sources'][j-1]
                        output_text += f"Source: {src['source']} | Score: {src['score']}\n"
                    output_text += "-" * 20 + "\n"
                    output_text += chunk.strip() + "\n"
                    output_text += "-" * 20 + "\n"
                        
            output_text += f"\nHighest Confidence Score: {result['confidence']}\n"
            output_text += "="*80 + "\n\n"
                    
        f.write(output_text)
        print(output_text)

In [16]:
#9. evaluation execution with test query array 

test_queries = [
    "How many people work in the Great Barrier Reef industry in the whole of Australia?",
    "How many people work locally at the Great Barrier Reef?",
    "How many persons in Queensland work at the Great Barrier Reef?",
    "What is the value of the gross product which the Great Barrier Reef generates for Queensland? ",
    "What are the symptoms of the climate change at the Great Barrier Reef?",

    "Whats the geografical range of the Great Barrier Reef?",
    "How big is the Great Barrier Reef?",
    "Does the licence for ocean fishing cost a fee?",
    "How much will the Earth warm up?",
    "How will the climate change affect the tourism?",

    "How many visits does the Great Barrier Reef get on average per year?",
    "What does happen to the fish if the reef gets in a bad condition?",
    "What happens to the octopus?",
    "Where is Nemo?",
    "Can we calculate the total population of Australia?"
]       

evaluate_test_queries(test_queries, rag_retriever, llm, filename="../data/evaluation/evaluation__all_chunks_output.txt")

________ printed chunks _________
Question 1: How many people work in the Great Barrier Reef industry in the whole of Australia?

Retrieving documents for query: 'How many people work in the Great Barrier Reef industry in the whole of Australia?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]


Retrieved 5 documents above the score threshold of 0.0
Generated answer:
________________________________________
About 56,000 persons (full-time equivalent basis).
________________________________________
Question 2: How many people work locally at the Great Barrier Reef?

Retrieving documents for query: 'How many people work locally at the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]


Retrieved 5 documents above the score threshold of 0.0
Generated answer:
________________________________________
About 66,000 persons (full-time equivalent basis) work locally in the Great Barrier Reef region, with tourism dominating these contributions.
________________________________________
Question 3: How many persons in Queensland work at the Great Barrier Reef?

Retrieving documents for query: 'How many persons in Queensland work at the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


...

Retrieved 5 documents above the score threshold of 0.0


Generated answer:
________________________________________
There is no mention of Nemo in the provided context.
________________________________________
Question 15: Can we calculate the total population of Australia?

Retrieving documents for query: 'Can we calculate the total population of Australia?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.07it/s]

Retrieved 5 documents above the score threshold of 0.0


Generated answer:
________________________________________
No, the provided context does not mention the total population of Australia. It only provides information on the economic value and employment figures for the Great Barrier Reef Marine Park (GBRMP) in Queensland and Australia.
________________________________________

Question 1:
How many people work in the Great Barrier Reef industry in the whole of Australia?

Answer:
----------------------------------------
About 56,000 persons (full-time equivalent basis).
----------------------------------------

Retrieved chunks:

[Chunk 1]
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf | Score: 0.562
--------------------
For employment (full time equivalent basis), about 44,000 persons. The corresponding estimates for Queensland are: For value-added, around $4.5 billion per annum. For gross product, around $5.4 billion per annum. For employment (full time equivalent basis), about